# Week 5 – Werkcollege 7: geopandas, Folium en Bot v3

Je pokerbot krijgt deze week een bluf-parameter en een locatie. Bot v3 bluft soms met een zwakke hand, en je koppelt een campus of stad aan je bot zodat de klas straks op een kaart te zien is.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Werkcollege | 40 min | Geopandas basis + Folium |
| Werkcollege | 15 min | Uitleg pokerbot-upgrade Week 5 |
| Werkcollege | 35 min | Zelf starten: Bot v3 skeleton |
| Huiswerk | 25 min | Bot v3 afmaken |
| Huiswerk | 10 min | Locatie kiezen |
| Huiswerk | 25 min | Folium-kaart met alle bots |
| Huiswerk | 15 min | Lokaal testen tegen je Week 3-bot |
| Huiswerk | 15 min | Inleveren via de API (met bluf_kans en locatie) |
| Huiswerk | 10 min | Reflectievragen |
| **Totaal huiswerk** | **± 100 min** | Deadline: woensdag 09:00 |


## Deel 1 — Geopandas en Folium (± 40 min)

### 1. Een GeoDataFrame bouwen (12 min)

Een gewone pandas-tabel met een lat/lon-kolom is nog geen geografische data — daarvoor heb je een geometrie-kolom nodig. `geopandas.points_from_xy()` zet losse lon/lat-waarden om in punten.


In [ ]:
import geopandas
import pandas as pd

voorbeeld_locaties = pd.DataFrame({
    "naam": ["Amsterdam", "Rotterdam", "Utrecht", "Groningen"],
    "lat": [52.37, 51.92, 52.09, 53.22],
    "lon": [4.90, 4.48, 5.12, 6.57],
})

geometrie = geopandas.points_from_xy(voorbeeld_locaties["lon"], voorbeeld_locaties["lat"])
bot_locaties = geopandas.GeoDataFrame(voorbeeld_locaties, geometry=geometrie, crs="EPSG:4326")
bot_locaties


🤔 Wat is het verschil tussen de kolommen `lat`/`lon` en de kolom `geometry`? Print het datatype van `bot_locaties["geometry"]` om te checken.


### 2. Optioneel: een wereldkaart als achtergrond (8 min)

Dit stapje downloadt een kaartbestand van het internet — sla 'm over als je geen verbinding hebt, de rest van het werkcollege heeft dit niet nodig.


In [ ]:
try:
    import geodatasets
    import matplotlib.pyplot as plt

    pad = geodatasets.get_path("naturalearth.land")
    wereld = geopandas.read_file(pad)

    ax = wereld.plot(color="#DDDDDD", edgecolor="white", figsize=(8, 8))
    bot_locaties.plot(ax=ax, color="#E4572E", markersize=40)
    ax.set_title("Voorbeeld bot-locaties op de wereldkaart")
    plt.show()
except Exception as e:
    print("Kon de wereldkaart niet downloaden (geen internet?):", e)
    print("Geen probleem — de GeoDataFrame uit stap 1 is de kernvaardigheid, niet deze achtergrond.")


### 3. Folium: een interactieve kaart (20 min)

Folium tekent op echte kaarttegels (OpenStreetMap) in plaats van een statische afbeelding — je kan inzoomen, slepen, en op markers klikken.


In [ ]:
import folium

kaart = folium.Map(location=[52.1, 5.1], zoom_start=7)

for _, rij in bot_locaties.iterrows():
    folium.Marker(
        location=[rij["lat"], rij["lon"]],
        popup=rij["naam"],
        tooltip=rij["naam"],
    ).add_to(kaart)

kaart


`popup` verschijnt pas na een klik, `tooltip` al bij hover. Verander bij één marker de popup-tekst naar iets met meer detail (bv. een fictieve stack), en kijk wat het verschil is met de tooltip.


---

## Deel 2 — Pokerbot Upgrade Week 5: bluffen en locatie (15 min)

Bot v3 krijgt een `bluf_kans`: de kans dat hij met een zwakke hand toch raiset, om tegenstanders te misleiden. Daarnaast koppel je een locatie aan je inzending — niet omdat je bot ergens speelt, maar zodat de klas straks op een kaart kan zien waar iedereen vandaan komt, gekleurd op winst of verlies.

Bij het inleveren geef je vanaf nu ook `bluf_kans` (een getal tussen 0 en 1) en `locatie` (`lat`, `lon`, `plaatsnaam`) mee. Ontbreekt een van beide, of klopt het niet, dan krijg je een duidelijke foutmelding terug.


## Deel 3 — Zelf starten: Bot v3 skeleton (35 min)

Schrijf `kies_actie(hand, stack, strategie, bluf_kans)`. Bepaal eerst met `random.random() < bluf_kans` of dit een bluf-hand wordt; reageer daarna zoals in Week 3, met dat verschil dat een bluf een `raise` forceert ook al is de hand zwak.


In [ ]:
import random

def kies_actie(hand, stack, strategie, bluf_kans):
    """hand, stack, strategie: zie Week 3. bluf_kans: kans (0-1) dat je bluft met een zwakke hand."""
    # jouw logica hier
    pass

kies_actie(["7", "2"], 1000, "tight", 0.3)  # test


---

## Deel 4 — Bot v3 afmaken (huiswerk, ± 25 min)

Werk je `kies_actie` verder uit. Test 'm met een paar combinaties, en zet vooraf een `random.seed()` zodat je je eigen testresultaten kan reproduceren.


In [ ]:
random.seed(1)

test_gevallen = [
    (["7", "2"], 1000, "tight", 0.0),
    (["7", "2"], 1000, "tight", 1.0),
    (["A", "A"], 1000, "tight", 0.5),
]
for hand, stack, strategie, bluf_kans in test_gevallen:
    print(hand, stack, strategie, bluf_kans, "->", kies_actie(hand, stack, strategie, bluf_kans))


🤔 Bij `bluf_kans=1.0` en een zwakke hand zou je functie altijd moeten raisen. Klopt dat met wat je ziet? Zo niet, waar zit de fout?


## Deel 5 — Locatie kiezen (huiswerk, ± 10 min)

Kies een lat/lon voor je eigen bot: je campus, je woonplaats, of iets anders dat je leuk vindt. Een paar Nederlandse steden als referentie:


In [ ]:
NEDERLANDSE_STEDEN = {
    "Amsterdam": (52.37, 4.90),
    "Rotterdam": (51.92, 4.48),
    "Den Haag": (52.08, 4.31),
    "Utrecht": (52.09, 5.12),
    "Eindhoven": (51.44, 5.48),
    "Groningen": (53.22, 6.57),
    "Nijmegen": (51.84, 5.85),
    "Tilburg": (51.56, 5.09),
}

MIJN_PLAATS = "Amsterdam"  # pas aan
mijn_lat, mijn_lon = NEDERLANDSE_STEDEN[MIJN_PLAATS]


## Deel 6 — Folium-kaart met alle bots (huiswerk, ± 25 min)

Haal de locaties van de hele klas op bij de API. Zolang het toernooi van deze week nog niet gedraaid is, is `eindstand` overal `null` — dan kleur je alle markers voorlopig hetzelfde. Woensdag (Werkcollege 8) is dat gevuld.


In [ ]:
import requests

API_URL = "https://poker-analytics-api.onrender.com"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

response = requests.get(
    f"{API_URL}/locaties/5",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
locaties = response.json()
locaties


In [ ]:
kaart_klas = folium.Map(location=[52.1, 5.1], zoom_start=7)

for bot in locaties:
    if bot["eindstand"] is None:
        kleur = "gray"
    elif bot["eindstand"] >= 1000:
        kleur = "green"
    else:
        kleur = "red"

    folium.CircleMarker(
        location=[bot["lat"], bot["lon"]],
        radius=8,
        color=kleur,
        fill=True,
        fill_color=kleur,
        popup=f"{bot['student_id']} ({bot['plaatsnaam']}) — eindstand: {bot['eindstand']}",
        tooltip=bot["student_id"],
    ).add_to(kaart_klas)

kaart_klas


🎨 Dit is bewust nog een simpele versie: één kleur, één simpele popup. Volgende Visual Maandag (Week 6) ga je hier gelaagde informatie in stoppen zonder de kaart vol te proppen.


## Deel 7 — Lokaal testen tegen je Week 3-bot (huiswerk, ± 15 min)


In [ ]:
import sys
sys.path.append('.')
from mijn_bot_week3 import kies_actie as bot_v2

for hand in [["7", "2"], ["A", "A"], ["K", "Q"]]:
    print(hand, "v2:", bot_v2(hand, 1000, "tight"), "| v3:", kies_actie(hand, 1000, "tight", 0.3))


## Deel 8 — Inleveren via de API (huiswerk, ± 15 min)


In [ ]:
from _hulpfuncties import export_chart_info, lever_in

MIJN_STRATEGIE = "tight"
MIJN_BLUF_KANS = 0.3
MIJN_LOCATIE = {"lat": mijn_lat, "lon": mijn_lon, "plaatsnaam": MIJN_PLAATS}

with open("mijn_bot_week5.py") as f:
    bot_code = f.read()

chart_info = export_chart_info(
    kaart_klas,
    titel="vul hier een echte actietitel in",
    library="folium",
)

resultaat = lever_in(
    STUDENT_ID, TOKEN, week=5, bot_code=bot_code, chart_info=chart_info,
    strategie=MIJN_STRATEGIE, bluf_kans=MIJN_BLUF_KANS, locatie=MIJN_LOCATIE,
)
resultaat


---

## Reflectievragen

🤔 Waarom moet `bluf_kans` een kans zijn (0 tot 1) en geen vast "ja/nee"? Wat zou je bot verliezen als bluffen altijd of nooit gebeurde?

💡 Je locatie heeft niets te maken met hoe je bot speelt. Waarom voegt de docent 'm dan toch toe aan de inzending?

🤔 In Deel 6 kleurde je markers grijs zolang de eindstand nog `null` was. Wat was er misgegaan als je code ervan uitging dat `eindstand` altijd een getal is?

🎨 Kijk naar je kaart uit Deel 6. Als iemand 'm voor het eerst ziet: kan diegene zonder uitleg zien wat groen/rood/grijs betekent?

💡 Je bot heeft nu 3 versies (v1, v2, v3) met steeds meer parameters. Welke volgende parameter zou jij toevoegen als je nog een Week 7 had?

---

### Vooruitblik: Werkcollege 8 (woensdag)

Je Bot v3 speelt tegen de klas. Je krijgt de resultaten, en je kaart kleurt zichzelf in op basis van winst en verlies.
